# ECERC Limitation 2, Experiment 5: a fine-tuned LLM expert fused with ECERC

Same design as Experiment 4, with a much stronger expert: **Qwen2.5-7B-Instruct**, fine-tuned with QLoRA (4-bit base, LoRA adapters, classification head) on each utterance plus the preceding dialogue. It uses past context only, ECERC's task setting. Two seeds train **in parallel, one per T4**.

Its predictions are combined with each released ECERC checkpoint through one weight chosen on dev: 3 checkpoints × 2 seeds = 6 pairs.

The test runs **once**, only if a cross-fitted dev estimate shows at least +0.5 macro-F1, after every decision is frozen. This is the third look at the MELD test set, so a multiplicity-adjusted interval is reported as well.

Expect roughly 3 to 6 hours on Kaggle **GPU T4 x2**, with Internet **On**. Run it with **Save Version → Save & Run All (Commit)**.

In [ ]:
# ===== Configuration: leave as is. (Set TRUST_DATA_PICKLE = True only if the data cell asks you to.) =====
import os

TRUST_DATA_PICKLE = False

if os.path.isdir('/kaggle/working'):
    WORK_DIR = '/kaggle/working/ecerc_l2'
elif os.path.isdir('/content'):
    WORK_DIR = '/content/ecerc_l2'
else:
    WORK_DIR = os.path.abspath('ecerc_l2_work')
HF = 'https://huggingface.co/zt-ai/ECERC/resolve/main/MELD/'
CONFIG = dict(
    work_dir=WORK_DIR,
    data_file_id='1IQokygWgjaZuSXvCsfIcUzj1Gu69hDCb',            # authors' Google Drive (linked from the official ECERC README)
    data_filename='meld_emotion_semantic_features_roberta.pkl',
    checkpoints=[
        dict(name='C1', url=HF + 'ECERC_MODEL.pkl', sha256='0f44a6e546592a3f7843342c0f72dcfa591eab1f99cdbd8a7d4ea7aed607441c'),
        dict(name='C2', url=HF + 'ECERC_MODEL_2.pkl', sha256='47472e9ed138934737d3516be3bd630a16edad14971fe29ce5002e1d7bbdf28c'),
        dict(name='C3', url=HF + 'ECERC_MODEL_3.pkl', sha256='d5104e06fb050410e0720b9de4310474339cd2494eb98e2f57f9df89a609bf5c'),
    ],
    trust_data_pickle=TRUST_DATA_PICKLE,
    expected_split_utterances=dict(train=9989, valid=1109, test=2610),
    expected_split_dialogues=dict(train=[1038, 1039], valid=114, test=280),
    min_dev_acc=0.55,
    llm_model='Qwen/Qwen2.5-7B-Instruct',   # Apache-2.0, no access request needed
    llm_quantize=True,                      # 4-bit QLoRA so a 7B model trains on one 16 GB T4
    llm_seeds=[1, 2],                       # one per T4; they train in parallel
    llm_epochs=1, llm_lr=1e-4, llm_micro_batch=8, llm_grad_accum=2, llm_warmup=0.05,
    llm_eval_every=0.34, llm_patience=2, llm_eval_batch=16, llm_max_len=256,
    llm_max_minutes=420,                    # hard cap on training time per seed (keeps the run inside Kaggle's 12 h)
    llm_lora_r=16, llm_lora_alpha=32, llm_lora_dropout=0.05,
    llm_lora_targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    llm_min_dev_mf1=35.0,                   # one retry for a seed whose best dev macro-F1 stays below this
    gate_min_gain=0.5,                      # the test runs only if the cross-fitted dev gain is >= 0.5
    n_folds=5, fold_seed=12345, power_B=2000, bootstrap_B=10000, boot_seed=20261013,
    poll_seconds=30, stagger_seconds=120,   # the second worker starts 2 minutes after the first
    offline=False,                          # True only for the local smoke test
)
os.makedirs(WORK_DIR, exist_ok=True)
print('Working directory:', WORK_DIR)

## 1. Code
The four original ECERC files (verbatim) and the analysis code, embedded as base64 and checked by SHA-256. Readable copies are in `research/limitation2/`.

In [ ]:
# Writes the verbatim ECERC files and the analysis code, checks their SHA-256, then sets up.
import base64, hashlib, importlib, sys

CODE_DIR = os.path.join(CONFIG['work_dir'], 'code')
os.makedirs(CODE_DIR, exist_ok=True)
CODE_FILES = {  # name: (sha256, base64 content)
    'model.py': ('a09dbe3aa6fd662237fb9833a0544887e9441e46a08783ca39b363d97d6ccc32', '''
aW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKZnJvbSB0b3JjaCBp
bXBvcnQgbm4KaW1wb3J0IG51bXB5IGFzIG5wCgoKY2xhc3MgUG9zaXRpb25hbEVuY29kaW5nKG5u
Lk1vZHVsZSk6CgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGFyZ3MsIGRfaGlkLCBuX3Bvc2l0aW9u
PTIwMCk6CiAgICAgICAgc3VwZXIoUG9zaXRpb25hbEVuY29kaW5nLCBzZWxmKS5fX2luaXRfXygp
CgogICAgICAgICMgTm90IGEgcGFyYW1ldGVyCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIo
J3Bvc190YWJsZScsIHNlbGYuX2dldF9zaW51c29pZF9lbmNvZGluZ190YWJsZShuX3Bvc2l0aW9u
LCBkX2hpZCkpCiAgICAgICAgc2VsZi5hcmdzID0gYXJncwogICAgZGVmIF9nZXRfc2ludXNvaWRf
ZW5jb2RpbmdfdGFibGUoc2VsZiwgbl9wb3NpdGlvbiwgZF9oaWQpOgogICAgICAgICcnJyBTaW51
c29pZCBwb3NpdGlvbiBlbmNvZGluZyB0YWJsZSAnJycKCiAgICAgICAgIyBUT0RPOiBtYWtlIGl0
IHdpdGggdG9yY2ggaW5zdGVhZCBvZiBudW1weQoKICAgICAgICBkZWYgZ2V0X3Bvc2l0aW9uX2Fu
Z2xlX3ZlYyhwb3NpdGlvbik6CiAgICAgICAgICAgIHJldHVybiBbcG9zaXRpb24gLyBucC5wb3dl
cigxMDAwMCwgMiAqIChoaWRfaiAvLyAyKSAvIGRfaGlkKSBmb3IgaGlkX2ogaW4gcmFuZ2UoZF9o
aWQpXQoKICAgICAgICBzaW51c29pZF90YWJsZSA9IG5wLmFycmF5KFtnZXRfcG9zaXRpb25fYW5n
bGVfdmVjKHBvc19pKSBmb3IgcG9zX2kgaW4gcmFuZ2Uobl9wb3NpdGlvbildKQogICAgICAgIHNp
bnVzb2lkX3RhYmxlWzosIDA6OjJdID0gbnAuc2luKHNpbnVzb2lkX3RhYmxlWzosIDA6OjJdKSAg
IyBkaW0gMmkKICAgICAgICBzaW51c29pZF90YWJsZVs6LCAxOjoyXSA9IG5wLmNvcyhzaW51c29p
ZF90YWJsZVs6LCAxOjoyXSkgICMgZGltIDJpKzEKCiAgICAgICAgcmV0dXJuIHRvcmNoLkZsb2F0
VGVuc29yKHNpbnVzb2lkX3RhYmxlKS51bnNxdWVlemUoMCkKCiAgICBkZWYgZm9yd2FyZChzZWxm
LCB4KToKICAgICAgICByZXR1cm4geCArIDAuICogc2VsZi5wb3NfdGFibGVbOiwgOnguc2l6ZSgx
KSwgOnguc2l6ZSgyKV0uY2xvbmUoKS5kZXRhY2goKQoKCmNsYXNzIFNjYWxlZERvdFByb2R1Y3RB
dHRlbnRpb24obm4uTW9kdWxlKToKICAgICcnJyBTY2FsZWQgRG90LVByb2R1Y3QgQXR0ZW50aW9u
ICcnJwoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB0ZW1wZXJhdHVyZSwgYXR0bl9kcm9wb3V0PTAu
MSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi50ZW1wZXJhdHVyZSA9
IHRlbXBlcmF0dXJlCiAgICAgICAgc2VsZi5kcm9wb3V0ID0gbm4uRHJvcG91dChhdHRuX2Ryb3Bv
dXQpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgcSwgaywgdiwgbWFzaz1Ob25lKToKICAgICAgICAi
IiIKICAgICAgICBxLCBrLCB2OiBbYiB4IG4geCBscSB4IGRfa10sIFtiIHggbiB4IGxxIHggZF9r
XSwgW2IgeCBuIHggbHEgeCBkX3ZdCiAgICAgICAgIiIiCiAgICAgICAgYXR0biA9IHRvcmNoLm1h
dG11bChxIC8gc2VsZi50ZW1wZXJhdHVyZSwgay50cmFuc3Bvc2UoMiwgMykpCgogICAgICAgIGlm
IG1hc2sgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGF0dG4gPSBhdHRuLm1hc2tlZF9maWxsKG1h
c2sgPT0gMCwgLTFlOSkKCiAgICAgICAgYXR0biA9IHNlbGYuZHJvcG91dChGLnNvZnRtYXgoYXR0
biwgZGltPS0xKSkKICAgICAgICBvdXRwdXQgPSB0b3JjaC5tYXRtdWwoYXR0biwgdikKCiAgICAg
ICAgcmV0dXJuIG91dHB1dCwgYXR0bgoKCmNsYXNzIE11bHRpSGVhZEF0dGVudGlvbihubi5Nb2R1
bGUpOgogICAgJycnIE11bHRpLUhlYWQgQXR0ZW50aW9uIG1vZHVsZSAnJycKCiAgICBkZWYgX19p
bml0X18oc2VsZiwgbl9oZWFkLCBkX21vZGVsLCBkX2ssIGRfdiwgZHJvcG91dD0wLjEpOgogICAg
ICAgIHN1cGVyKCkuX19pbml0X18oKQoKICAgICAgICBzZWxmLm5faGVhZCA9IG5faGVhZAogICAg
ICAgIHNlbGYuZF9rID0gZF9rCiAgICAgICAgc2VsZi5kX3YgPSBkX3YKICAgICAgICBzZWxmLndf
cXMgPSBubi5MaW5lYXIoZF9tb2RlbCwgbl9oZWFkICogZF9rLCBiaWFzPUZhbHNlKQogICAgICAg
IHNlbGYud19rcyA9IG5uLkxpbmVhcihkX21vZGVsLCBuX2hlYWQgKiBkX2ssIGJpYXM9RmFsc2Up
CiAgICAgICAgc2VsZi53X3ZzID0gbm4uTGluZWFyKGRfbW9kZWwsIG5faGVhZCAqIGRfdiwgYmlh
cz1GYWxzZSkKICAgICAgICBzZWxmLmZjID0gbm4uTGluZWFyKG5faGVhZCAqIGRfdiwgZF9tb2Rl
bCwgYmlhcz1GYWxzZSkKCiAgICAgICAgc2VsZi5hdHRlbnRpb24gPSBTY2FsZWREb3RQcm9kdWN0
QXR0ZW50aW9uKHRlbXBlcmF0dXJlPWRfayAqKiAwLjUsIGF0dG5fZHJvcG91dD1kcm9wb3V0KQoK
ICAgICAgICBzZWxmLmRyb3BvdXQgPSBubi5Ecm9wb3V0KGRyb3BvdXQpCiAgICAgICAgc2VsZi5s
YXllcl9ub3JtID0gbm4uTGF5ZXJOb3JtKGRfbW9kZWwsIGVwcz0xZS02KQoKICAgIGRlZiBmb3J3
YXJkKHNlbGYsIHEsIGssIHYsIG1hc2s9Tm9uZSk6CiAgICAgICAgZF9rLCBkX3YsIG5faGVhZCA9
IHNlbGYuZF9rLCBzZWxmLmRfdiwgc2VsZi5uX2hlYWQKICAgICAgICBzel9iLCBsZW5fcSwgbGVu
X2ssIGxlbl92ID0gcS5zaXplKDApLCBxLnNpemUoMSksIGsuc2l6ZSgxKSwgdi5zaXplKDEpCgog
ICAgICAgIHJlc2lkdWFsID0gcQoKICAgICAgICAjIFBhc3MgdGhyb3VnaCB0aGUgcHJlLWF0dGVu
dGlvbiBwcm9qZWN0aW9uOiBiIHggbHEgeCAobipkdikKICAgICAgICAjIFNlcGFyYXRlIGRpZmZl
cmVudCBoZWFkczogYiB4IGxxIHggbiB4IGR2CiAgICAgICAgcSA9IHNlbGYud19xcyhxKS52aWV3
KHN6X2IsIGxlbl9xLCBuX2hlYWQsIGRfaykKICAgICAgICBrID0gc2VsZi53X2tzKGspLnZpZXco
c3pfYiwgbGVuX2ssIG5faGVhZCwgZF9rKQogICAgICAgIHYgPSBzZWxmLndfdnModikudmlldyhz
el9iLCBsZW5fdiwgbl9oZWFkLCBkX3YpCgogICAgICAgICMgVHJhbnNwb3NlIGZvciBhdHRlbnRp
b24gZG90IHByb2R1Y3Q6IGIgeCBuIHggbHEgeCBkdgogICAgICAgIHEsIGssIHYgPSBxLnRyYW5z
cG9zZSgxLCAyKSwgay50cmFuc3Bvc2UoMSwgMiksIHYudHJhbnNwb3NlKDEsIDIpCgogICAgICAg
IGlmIG1hc2sgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG1hc2sgPSBtYXNrLnVuc3F1ZWV6ZSgx
KSAgIyBGb3IgaGVhZCBheGlzIGJyb2FkY2FzdGluZy4KCiAgICAgICAgcSwgYXR0biA9IHNlbGYu
YXR0ZW50aW9uKHEsIGssIHYsIG1hc2s9bWFzaykKCiAgICAgICAgIyBUcmFuc3Bvc2UgdG8gbW92
ZSB0aGUgaGVhZCBkaW1lbnNpb24gYmFjazogYiB4IGxxIHggbiB4IGR2CiAgICAgICAgIyBDb21i
aW5lIHRoZSBsYXN0IHR3byBkaW1lbnNpb25zIHRvIGNvbmNhdGVuYXRlIGFsbCB0aGUgaGVhZHMg
dG9nZXRoZXI6IGIgeCBscSB4IChuKmR2KQogICAgICAgIHEgPSBxLnRyYW5zcG9zZSgxLCAyKS5j
b250aWd1b3VzKCkudmlldyhzel9iLCBsZW5fcSwgLTEpCiAgICAgICAgcSA9IHNlbGYuZHJvcG91
dChzZWxmLmZjKHEpKQogICAgICAgIHEgKz0gcmVzaWR1YWwKCiAgICAgICAgcSA9IHNlbGYubGF5
ZXJfbm9ybShxKQoKICAgICAgICByZXR1cm4gcQoKCmNsYXNzIFBvc2l0aW9ud2lzZUZlZWRGb3J3
YXJkKG5uLk1vZHVsZSk6CiAgICAnJycgQSB0d28tZmVlZC1mb3J3YXJkLWxheWVyIG1vZHVsZSAn
JycKCiAgICBkZWYgX19pbml0X18oc2VsZiwgZF9pbiwgZF9oaWQsIGRyb3BvdXQ9MC4xKToKICAg
ICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLndfMSA9IG5uLkxpbmVhcihkX2lu
LCBkX2hpZCkgICMgcG9zaXRpb24td2lzZQogICAgICAgIHNlbGYud18yID0gbm4uTGluZWFyKGRf
aGlkLCBkX2luKSAgIyBwb3NpdGlvbi13aXNlCiAgICAgICAgc2VsZi5sYXllcl9ub3JtID0gbm4u
TGF5ZXJOb3JtKGRfaW4sIGVwcz0xZS02KQogICAgICAgIHNlbGYuZHJvcG91dCA9IG5uLkRyb3Bv
dXQoZHJvcG91dCkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICByZXNpZHVhbCA9
IHgKICAgICAgICB4ID0gc2VsZi53XzIoRi5yZWx1KHNlbGYud18xKHgpKSkKICAgICAgICB4ID0g
c2VsZi5kcm9wb3V0KHgpCiAgICAgICAgeCArPSByZXNpZHVhbAoKICAgICAgICB4ID0gc2VsZi5s
YXllcl9ub3JtKHgpCgogICAgICAgIHJldHVybiB4CgoKY2xhc3MgRW5jb2RlckxheWVyKG5uLk1v
ZHVsZSk6CiAgICAnJycgQ29tcG9zZSB3aXRoIHR3byBsYXllcnMgJycnCgogICAgZGVmIF9faW5p
dF9fKHNlbGYsIG5faGVhZCwgZF9tb2RlbCwgZF9rLCBkX3YsIGRyb3BvdXQ9MC4xKToKICAgICAg
ICBzdXBlcihFbmNvZGVyTGF5ZXIsIHNlbGYpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnNsZl9h
dHRuID0gTXVsdGlIZWFkQXR0ZW50aW9uKG5faGVhZCwgZF9tb2RlbCwgZF9rLCBkX3YsIGRyb3Bv
dXQ9ZHJvcG91dCkKICAgICAgICBzZWxmLnBvc19mZm4gPSBQb3NpdGlvbndpc2VGZWVkRm9yd2Fy
ZChkX21vZGVsLCBkX21vZGVsICogMiwgZHJvcG91dD1kcm9wb3V0KQoKICAgIGRlZiBmb3J3YXJk
KHNlbGYsIHEsIGssIHYsIG1hc2s9Tm9uZSk6CiAgICAgICAgZW5jX291dHB1dCA9IHNlbGYuc2xm
X2F0dG4ocSwgaywgdiwgbWFzaz1tYXNrKQogICAgICAgIGVuY19vdXRwdXQgPSBzZWxmLnBvc19m
Zm4oZW5jX291dHB1dCkKICAgICAgICByZXR1cm4gZW5jX291dHB1dAoKCmNsYXNzIEVuY29kZXIo
bm4uTW9kdWxlKToKICAgICcnJyBBIGVuY29kZXIgbW9kZWwgd2l0aCBzZWxmIGF0dGVudGlvbiBt
ZWNoYW5pc20uICcnJwoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICAgICAgc2VsZiwgbl9sYXll
ciwgbl9oZWFkLCBkX21vZGVsLCBkX2ssIGRfdiwgZHJvcG91dD0wLjEsIHNjYWxlX2VtYj1GYWxz
ZSk6CgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZHJvcG91dCA9IG5u
LkRyb3BvdXQocD1kcm9wb3V0KQogICAgICAgIHNlbGYubGF5ZXJfc3RhY2sgPSBubi5Nb2R1bGVM
aXN0KFsKICAgICAgICAgICAgRW5jb2RlckxheWVyKG5faGVhZCwgZF9tb2RlbCwgZF9rLCBkX3Ys
IGRyb3BvdXQ9ZHJvcG91dCkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2Uobl9sYXllcildKQog
ICAgICAgICMgc2VsZi5sYXllcl9ub3JtX3EgPSBubi5MYXllck5vcm0oZF9tb2RlbCwgZXBzPTFl
LTYpCiAgICAgICAgIyBzZWxmLmxheWVyX25vcm1fayA9IG5uLkxheWVyTm9ybShkX21vZGVsLCBl
cHM9MWUtNikKICAgICAgICAjIHNlbGYubGF5ZXJfbm9ybV92ID0gbm4uTGF5ZXJOb3JtKGRfbW9k
ZWwsIGVwcz0xZS02KQogICAgICAgIHNlbGYuc2NhbGVfZW1iID0gc2NhbGVfZW1iCiAgICAgICAg
c2VsZi5kX21vZGVsID0gZF9tb2RlbAoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHEsIGssIHYsIG1v
ZGUsIG1hc2s9Tm9uZSk6CgogICAgICAgICMgLS0gRm9yd2FyZAogICAgICAgIGlmIHNlbGYuc2Nh
bGVfZW1iOgogICAgICAgICAgICBxICo9IHNlbGYuZF9tb2RlbCAqKiAwLjUKICAgICAgICAgICAg
ayAqPSBzZWxmLmRfbW9kZWwgKiogMC41CiAgICAgICAgICAgIHYgKj0gc2VsZi5kX21vZGVsICoq
IDAuNQogICAgICAgIGZvciBpZHgsIGVuY19sYXllciBpbiBlbnVtZXJhdGUoc2VsZi5sYXllcl9z
dGFjayk6CiAgICAgICAgICAgIHEgPSBlbmNfbGF5ZXIocSwgaywgdiwgbWFzaz1tYXNrKQogICAg
ICAgICAgICBpZiBtb2RlID09ICJzZWxmIjoKICAgICAgICAgICAgICAgIGsgPSB2ID0gcQogICAg
ICAgIHJldHVybiBxCgoKY2xhc3MgTW9kYWxGaWx0ZXJWMihubi5Nb2R1bGUpOgogICAgJycnIEFu
IGVuY29kZXIgbW9kZWwgd2l0aCBzZWxmIGF0dGVudGlvbiBtZWNoYW5pc20uICcnJwoKICAgIGRl
ZiBfX2luaXRfXygKICAgICAgICAgICAgc2VsZiwgZF90LCBkX2EsIGRfdiwgaGlkZGVuKToKCiAg
ICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5wcm9qX3QgPSBubi5MaW5lYXIo
ZF90LCBoaWRkZW4pCiAgICAgICAgc2VsZi5wcm9qX2EgPSBubi5MaW5lYXIoZF9hLCBoaWRkZW4p
CiAgICAgICAgc2VsZi5wcm9qX3YgPSBubi5MaW5lYXIoZF92LCBoaWRkZW4pCgogICAgICAgIHNl
bGYucSA9IG5uLkxpbmVhcihoaWRkZW4sIGhpZGRlbikKICAgICAgICBzZWxmLmsgPSBubi5MaW5l
YXIoaGlkZGVuLCBoaWRkZW4pCgoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHQsIGEsIHYpOgogICAg
ICAgICIiIgoKICAgICAgICA6cGFyYW0gdDogKEIsIEwsIERfdCkKICAgICAgICA6cGFyYW0gYTog
IChCLCBMLCBEX2EpCiAgICAgICAgOnBhcmFtIHY6ICAoQiwgTCwgRF92KQogICAgICAgIDpyZXR1
cm46CiAgICAgICAgIiIiCiAgICAgICAgdCwgYSwgdiA9IHNlbGYucHJval90KHQpLCBzZWxmLnBy
b2pfYShhKSwgc2VsZi5wcm9qX3YodikKICAgICAgICAjIHQgPC0tIGEsIHYKICAgICAgICB3X3Qg
PSBGLnNpZ21vaWQoc2VsZi5xKHQpICsgc2VsZi5rKGEpICsgc2VsZi5rKHYpKQogICAgICAgIHdf
YSA9IEYuc2lnbW9pZChzZWxmLnEoYSkgKyBzZWxmLmsodCkgKyBzZWxmLmsodikpCiAgICAgICAg
d192ID0gRi5zaWdtb2lkKHNlbGYucSh2KSArIHNlbGYuayh0KSArIHNlbGYuayhhKSkKCiAgICAg
ICAgcmV0dXJuIHRvcmNoLmNhdChbd190ICogKHQpLCB3X2EgKiAoYSksIHdfdiAqICh2KV0sIGRp
bT0tMSkKCgpjbGFzcyBFQ0VSQyhubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGFy
Z3MsIGRfdCwgZF9hLCBkX3YsIGJhc2VfbGF5ZXI9MywgaW5wdXRfc2l6ZT1Ob25lLCBoaWRkZW5f
c2l6ZT1Ob25lLCBuX3NwZWFrZXJzPTIsCiAgICAgICAgICAgICAgICAgbl9jbGFzc2VzPTcsIGN1
ZGFfZmxhZz1GYWxzZSk6CiAgICAgICAgIiIiCiAgICAgICAgQ29udGV4dHVhbCBSZWFzb25pbmcg
TmV0d29yawogICAgICAgICIiIgoKICAgICAgICBzdXBlcihFQ0VSQywgc2VsZikuX19pbml0X18o
KQogICAgICAgIHNlbGYuYmFzZV9sYXllciA9IGJhc2VfbGF5ZXIKICAgICAgICBzZWxmLm5fc3Bl
YWtlcnMgPSBuX3NwZWFrZXJzCiAgICAgICAgc2VsZi5oaWRkZW5fc2l6ZSA9IGhpZGRlbl9zaXpl
CiAgICAgICAgc2VsZi5jdWRhX2ZsYWcgPSBjdWRhX2ZsYWcKICAgICAgICBzZWxmLmRfdCxzZWxm
LmRfYSwgc2VsZi5kX3YgPSBkX3QsIGRfYSwgZF92CgogICAgICAgIHNlbGYucG9zaXRpb25fZW5j
X2V2aSA9IFBvc2l0aW9uYWxFbmNvZGluZyhhcmdzLCBkX3QrZF9hK2Rfdiwgbl9wb3NpdGlvbj0y
MDApCgogICAgICAgIHNlbGYuZHJvcG91dDEgPSBubi5Ecm9wb3V0KHA9MC4pCiAgICAgICAgc2Vs
Zi5kcm9wb3V0MiA9IG5uLkRyb3BvdXQocD0wLjIpCiAgICAgICAgc2VsZi5sYXllcl9ub3JtID0g
bm4uTGF5ZXJOb3JtKGlucHV0X3NpemUsIGVwcz0xZS02KQoKICAgICAgICBzZWxmLm1vZGFsX2Zp
bHRlcmluZyA9IE1vZGFsRmlsdGVyVjIoZF90LCBkX2EsIGRfdiwgaGlkZGVuX3NpemUpCiAgICAg
ICAgaW5wdXRfc2l6ZSA9IGhpZGRlbl9zaXplICogMwogICAgICAgIHNlbGYuZW1vdGlvbl9lbmNv
ZGluZyA9IEVuY29kZXIobl9sYXllcj1iYXNlX2xheWVyLCBuX2hlYWQ9OCwgZF9tb2RlbD1pbnB1
dF9zaXplLCBkX2s9NjQsIGRfdj02NCkKICAgICAgICBzZWxmLnByb2pfZXZlID0gbm4uTGluZWFy
KGRfdCwgaGlkZGVuX3NpemUpCiAgICAgICAgc2VsZi5ldmVudF9lbmNvZGluZyA9IEVuY29kZXIo
bl9sYXllcj1iYXNlX2xheWVyLCBuX2hlYWQ9OCwgZF9tb2RlbD1oaWRkZW5fc2l6ZSwgZF9rPTY0
LCBkX3Y9NjQpCgogICAgICAgIHNlbGYuc2VsZl9jb250YWdpb25fYXR0ZW50aW9uID0gRW5jb2Rl
cihuX2xheWVyPWJhc2VfbGF5ZXIsIG5faGVhZD04LCBkX21vZGVsPWlucHV0X3NpemUsIGRfaz02
NCwgZF92PTY0KQogICAgICAgIHNlbGYuY3Jvc3NfZW1vdGlvbl9hdHRlbnRpb24gPSBFbmNvZGVy
KG5fbGF5ZXI9YmFzZV9sYXllciwgbl9oZWFkPTgsIGRfbW9kZWw9aW5wdXRfc2l6ZSwgZF9rPTY0
LCBkX3Y9NjQpCiAgICAgICAgc2VsZi5zZWxmX2V2ZW50X2F0dGVudGlvbiA9IEVuY29kZXIobl9s
YXllcj1iYXNlX2xheWVyLCBuX2hlYWQ9OCwgZF9tb2RlbD1oaWRkZW5fc2l6ZSwgZF9rPTY0LCBk
X3Y9NjQpCiAgICAgICAgc2VsZi5jcm9zc19ldmVudF9hdHRlbnRpb24gPSBFbmNvZGVyKG5fbGF5
ZXI9YmFzZV9sYXllciwgbl9oZWFkPTgsIGRfbW9kZWw9aGlkZGVuX3NpemUsIGRfaz02NCwgZF92
PTY0KQoKICAgICAgICBzZWxmLmdhdGVfcmVzZXQgPSBubi5MaW5lYXIoaW5wdXRfc2l6ZSwgaW5w
dXRfc2l6ZSkKICAgICAgICBzZWxmLmdhdGVfcmVzZXQyID0gbm4uTGluZWFyKGlucHV0X3NpemUs
IGlucHV0X3NpemUpCiAgICAgICAgc2VsZi5zbWF4X2ZjID0gbm4uTGluZWFyKGlucHV0X3NpemUg
KiAyICsgaW5wdXRfc2l6ZSAqIDIsIG5fY2xhc3NlcykKCgogICAgZGVmIGZvcndhcmQoc2VsZiwg
VV9lLCBVX3MsIHFtYXNrLCB1bWFzaywgc2VxX2xlbmd0aHMpOgogICAgICAgICMgRm9yIFZBVCBN
b2RhbGl0aWVzLgogICAgICAgICMgKGIsbCxkKSwgKGIsbCxwKQogICAgICAgIFVfZV8sIFVfc18s
IHFtYXNrXyA9IFVfZS50cmFuc3Bvc2UoMCwgMSksVV9zLnRyYW5zcG9zZSgwLCAxKSwgcW1hc2su
dHJhbnNwb3NlKDAsIDEpCiAgICAgICAgQiwgTCwgRCwgUCA9IFVfZV8uc2l6ZSgwKSwgVV9lXy5z
aXplKDEpLCBVX2VfLnNpemUoMiksIHFtYXNrXy5zaXplKDIpCgogICAgICAgIFVfZV8gPSBzZWxm
LnBvc2l0aW9uX2VuY19ldmkoVV9lXykKICAgICAgICBVX3NfID0gc2VsZi5wb3NpdGlvbl9lbmNf
ZXZpKFVfc18pCgogICAgICAgICIiIkNvbnN0cnVjdCBNYXNrcyIiIgogICAgICAgICMgSWRlbnRp
dHkgbWF0cml4CiAgICAgICAgaWRlbnRpdHlfbWFzayA9IHRvcmNoLmV5ZSh1bWFzay5zaXplKDEp
KS51bnNxdWVlemUoMCkuYm9vbCgpCgogICAgICAgICMgU3Vic2VxdWVuY2UgbWFzay4KICAgICAg
ICBzdWJtYXNrID0gKDEgLSB0b3JjaC50cml1KHRvcmNoLm9uZXMoKDEsIHVtYXNrLnNpemUoMSks
IHVtYXNrLnNpemUoMSkpLCBkZXZpY2U9dW1hc2suZGV2aWNlKSwgZGlhZ29uYWw9MSkpLmJvb2wo
KQogICAgICAgICMgUGFkZGluZyBtYXNrLgogICAgICAgIHBhZG1hc2sgPSB1bWFzay51bnNxdWVl
emUoLTIpLmJvb2woKQogICAgICAgICMgMi0+MTAKICAgICAgICB3ZWlnaHRzID0gMiAqKiB0b3Jj
aC5hcmFuZ2UoUCAtIDEsIC0xLCAtMSkuZmxvYXQoKQogICAgICAgIHdlaWdodHMgPSB3ZWlnaHRz
LmN1ZGEoKSBpZiBzZWxmLmN1ZGFfZmxhZyBlbHNlIHdlaWdodHMKICAgICAgICBkZWNpbWFsX3Rl
bnNvciA9IHRvcmNoLm1hdG11bChxbWFza18sIHdlaWdodHMpLmludCgpICAjIChCLCBMKQoKICAg
ICAgICAjIFNlbGYoU3BlYWtlcjJTcGVha2VyKSBtYXNrLgogICAgICAgIHNtYXNrID0gZGVjaW1h
bF90ZW5zb3IKICAgICAgICBzbWFzayA9IHNtYXNrLnVuc3F1ZWV6ZSgxKS5leHBhbmQoLTEsIEws
IC0xKSAjIFtCLEwsTF0KICAgICAgICBzbWFzayA9IChzbWFzay50cmFuc3Bvc2UoMSwyKSA9PSBz
bWFzaykuYm9vbCgpICMgW0IsTCxMXQogICAgICAgIHNtYXNrID0gc3VibWFzayAmIHBhZG1hc2sg
JiBzbWFzawogICAgICAgIHNtYXNrID0gc21hc2suY3VkYSgpIGlmIHNlbGYuY3VkYV9mbGFnIGVs
c2Ugc21hc2sKICAgICAgICAjIENyb3NzKFNwZWFrZXIyTGlzdGVuZXIpIG1hc2suCiAgICAgICAg
Y21hc2sgPSBkZWNpbWFsX3RlbnNvcgogICAgICAgIGNtYXNrID0gY21hc2sudW5zcXVlZXplKDEp
LmV4cGFuZCgtMSwgY21hc2suc2l6ZSgxKSwgLTEpICMgW0IsTCxMXQogICAgICAgIGNtYXNrID0g
KGNtYXNrLnRyYW5zcG9zZSgxLDIpICE9IGNtYXNrKS5ib29sKCkgIyBbQixMLExdCiAgICAgICAg
Y21hc2sgPSBzdWJtYXNrICYgcGFkbWFzayAmIGNtYXNrCiAgICAgICAgY21hc2sgPSBjbWFzay5j
dWRhKCkgaWYgc2VsZi5jdWRhX2ZsYWcgZWxzZSBjbWFzawoKICAgICAgICBpZiBzZWxmLmN1ZGFf
ZmxhZzoKICAgICAgICAgICAgaW1hc2ssIHNtYXNrLCBjbWFzaywgbWFzayA9IGlkZW50aXR5X21h
c2suY3VkYSgpLCBzbWFzay5jdWRhKCksIGNtYXNrLmN1ZGEoKSwgKHN1Ym1hc2sgJiBwYWRtYXNr
KS5jdWRhKCkKCiAgICAgICAgIiIiRXZpZGVuY2UgRW5jb2RpbmciIiIKICAgICAgICBldmlkZW5j
ZV9lbmMgPSBzZWxmLm1vZGFsX2ZpbHRlcmluZyhVX2VfWzosOiw6c2VsZi5kX3RdLFVfZV9bOiw6
LHNlbGYuZF90OihzZWxmLmRfdCtzZWxmLmRfYSldLFVfZV9bOiw6LChzZWxmLmRfdCtzZWxmLmRf
YSk6XSkgICMgb3JpZ2luYWwKICAgICAgICBldmlkZW5jZV9lbmMgPSBzZWxmLmRyb3BvdXQxKGV2
aWRlbmNlX2VuYykKICAgICAgICAiIiJFbW90aW9uIEVuY29kaW5nIGFzIENhdXNlcyIiIgogICAg
ICAgIGVtb3Rpb25fZW5jID0gc2VsZi5lbW90aW9uX2VuY29kaW5nKGV2aWRlbmNlX2VuYyxldmlk
ZW5jZV9lbmMsZXZpZGVuY2VfZW5jLCAic2VsZiIsIG1hc2spCgogICAgICAgICIiIkV2ZW50IEVu
Y29kaW5nIGFzIENhdXNlcyIiIgogICAgICAgIFVfc18gPSBzZWxmLnByb2pfZXZlKFVfc18pCiAg
ICAgICAgZXZlbnRfZW5jID0gc2VsZi5ldmVudF9lbmNvZGluZyhVX3NfLFVfc18sVV9zXywgInNl
bGYiLCBtYXNrKQoKICAgICAgICAiIiJDYXVzZSBGdXNpb246IEluZmx1ZW5jZWQgYnkgVHJpZ2dl
ciBFdmVudHMgTWVudGlvbmVkIGJ5IHRoZSBTcGVha2VyIFNlbGYuIiIiCiAgICAgICAgUl9zZWxm
X2V2ZW50ID0gc2VsZi5zZWxmX2V2ZW50X2F0dGVudGlvbihldmlkZW5jZV9lbmNbOiw6LDpzZWxm
LmhpZGRlbl9zaXplXSwgZXZlbnRfZW5jLCBldmVudF9lbmMsICJjcm9zcyIsIGltYXNrKQogICAg
ICAgIFJfc2VsZl9ldmVudCA9IHRvcmNoLmNhdChbUl9zZWxmX2V2ZW50LCBldmlkZW5jZV9lbmNb
Oiw6LHNlbGYuaGlkZGVuX3NpemU6XV0sIGRpbT0tMSkKCiAgICAgICAgIiIiQ2F1c2UgRnVzaW9u
OiBJbmZsdWVuY2VkIGJ5IFRyaWdnZXIgRXZlbnRzIE1lbnRpb25lZCBieSBPdGhlciBTcGVha2Vy
cy4iIiIKICAgICAgICBSX2Nyb3NzX2V2ZW50ID0gc2VsZi5jcm9zc19ldmVudF9hdHRlbnRpb24o
ZXZpZGVuY2VfZW5jWzosOiw6c2VsZi5oaWRkZW5fc2l6ZV0sIGV2ZW50X2VuYywgZXZlbnRfZW5j
LCAiY3Jvc3MiLCBjbWFzaykKICAgICAgICBSX2Nyb3NzX2V2ZW50ID0gdG9yY2guY2F0KFtSX2Ny
b3NzX2V2ZW50LCBldmlkZW5jZV9lbmNbOiw6LHNlbGYuaGlkZGVuX3NpemU6XV0sIGRpbT0tMSkK
CiAgICAgICAgIiIiQ2F1c2UgRnVzaW9uOiBTZWxmX0NvbnRhZ2lvbi4iIiIKICAgICAgICBSX2Nv
bnRhZ2lvbiA9IHNlbGYuc2VsZl9jb250YWdpb25fYXR0ZW50aW9uKGV2aWRlbmNlX2VuYywgZW1v
dGlvbl9lbmMsIGVtb3Rpb25fZW5jLCAiY3Jvc3MiLCBzbWFzayAmICh+aW1hc2spKSAgIyAoQiwg
TCwgRCkKCiAgICAgICAgIiIiQ2F1c2UgRnVzaW9uOiBJbmZsdWVuY2VkIGJ5IE90aGVyIFNwZWFr
ZXJzJyBFbW90aW9ucy4iIiIKICAgICAgICBSX2Nyb3NzX2Vtb3Rpb24gPSBzZWxmLmNyb3NzX2Vt
b3Rpb25fYXR0ZW50aW9uKGV2aWRlbmNlX2VuYywgZW1vdGlvbl9lbmMsIGVtb3Rpb25fZW5jLCAi
Y3Jvc3MiLCBjbWFzaykKCiAgICAgICAgIiIiR0FURSIiIgogICAgICAgIFJfbGlzID0gW1JfY29u
dGFnaW9uLCBSX2Nyb3NzX2Vtb3Rpb24sUl9zZWxmX2V2ZW50LFJfY3Jvc3NfZXZlbnRdCiAgICAg
ICAgd1JfbGlzPSBbRi5zaWdtb2lkKHNlbGYuZ2F0ZV9yZXNldChSKSkgKiAoUikgaWYgaWR4PDIg
ZWxzZSBGLnNpZ21vaWQoc2VsZi5nYXRlX3Jlc2V0MihSKSkgKiAoUikgZm9yIGlkeCwgUiBpbiBl
bnVtZXJhdGUoUl9saXMpXQogICAgICAgIFIgPSB0b3JjaC5jYXQod1JfbGlzLGRpbT0tMSkgICMg
KEIsIEwsIEQpCiAgICAgICAgaGlkZGVuID0gc2VsZi5zbWF4X2ZjKHNlbGYuZHJvcG91dDIoUikp
CgogICAgICAgIGxvZ19wcm9iID0gRi5sb2dfc29mdG1heChoaWRkZW4sIDIpLnRyYW5zcG9zZSgw
LDEpICAjIChMLCBCLCA0KjEwMjQpCiAgICAgICAgbG9nX3Byb2IgPSB0b3JjaC5jYXQoW2xvZ19w
cm9iWzosIGosIDpdWzpzZXFfbGVuZ3Roc1tqXV0gZm9yIGogaW4gcmFuZ2UobGVuKHNlcV9sZW5n
dGhzKSldKQogICAgICAgIHJldHVybiBsb2dfcHJvYgoKCg==
'''),
    'dataloader.py': ('121db840ea1d57c2814edb061909fcb9c31f9df944f73e1be42b5e6f8c1387ad', '''
aW1wb3J0IHRvcmNoCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YXNldApmcm9tIHRv
cmNoLm5uLnV0aWxzLnJubiBpbXBvcnQgcGFkX3NlcXVlbmNlCmltcG9ydCBwaWNrbGUsIHBhbmRh
cyBhcyBwZAppbXBvcnQgbnVtcHkKCgpjbGFzcyBNRUxEUm9iZXJ0YURhdGFzZXQoRGF0YXNldCk6
CgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHNwbGl0KToKICAgICAgICBzZWxmLnNwZWFrZXJzLCBz
ZWxmLmVtb3Rpb25fbGFiZWxzLCBzZWxmLnNlbnRpbWVudF9sYWJlbHMsXAogICAgICAgIHNlbGYu
ZXJvYmVydGExLCBzZWxmLmVyb2JlcnRhMiwgc2VsZi5lcm9iZXJ0YTMsIHNlbGYuZXJvYmVydGE0
LFwKICAgICAgICBzZWxmLnNyb2JlcnRhMSwgc2VsZi5zcm9iZXJ0YTIsIHNlbGYuc3JvYmVydGEz
LCBzZWxmLnNyb2JlcnRhNCxcCiAgICAgICAgc2VsZi52aWRlb0F1ZGlvLCBzZWxmLnZpZGVvVmlz
dWFsLFwKICAgICAgICBzZWxmLnNlbnRlbmNlcywgc2VsZi50cmFpbl9pZHMsIHNlbGYudGVzdF9p
ZHMsIHNlbGYudmFsaWRfaWRzXAogICAgICAgICAgICA9IHBpY2tsZS5sb2FkKG9wZW4oIi4uL2Rh
dGEvbWVsZC9tZWxkX2Vtb3Rpb25fc2VtYW50aWNfZmVhdHVyZXNfcm9iZXJ0YS5wa2wiLCAncmIn
KSwgZW5jb2Rpbmc9J2xhdGluMScpCgogICAgICAgIGlmIHNwbGl0ID09ICd0cmFpbic6CiAgICAg
ICAgICAgIHNlbGYua2V5cyA9IFt4IGZvciB4IGluIHNlbGYudHJhaW5faWRzXQogICAgICAgIGVs
aWYgc3BsaXQgPT0gJ3Rlc3QnOgogICAgICAgICAgICBzZWxmLmtleXMgPSBbeCBmb3IgeCBpbiBz
ZWxmLnRlc3RfaWRzXQogICAgICAgIGVsaWYgc3BsaXQgPT0gJ3ZhbGlkJzoKICAgICAgICAgICAg
c2VsZi5rZXlzID0gW3ggZm9yIHggaW4gc2VsZi52YWxpZF9pZHNdCgogICAgICAgIHNlbGYubGVu
ID0gbGVuKHNlbGYua2V5cykKCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaW5kZXgpOgogICAg
ICAgIHZpZCA9IHNlbGYua2V5c1tpbmRleF0KICAgICAgICByZXR1cm4gdG9yY2guRmxvYXRUZW5z
b3IobnVtcHkuYXJyYXkoc2VsZi5lcm9iZXJ0YTFbdmlkXSkpLFwKICAgICAgICAgICAgICAgdG9y
Y2guRmxvYXRUZW5zb3IobnVtcHkuYXJyYXkoc2VsZi5zcm9iZXJ0YTFbdmlkXSkpLFwKICAgICAg
ICAgICAgICAgdG9yY2guRmxvYXRUZW5zb3IobnVtcHkuYXJyYXkoc2VsZi52aWRlb1Zpc3VhbFt2
aWRdKSksXAogICAgICAgICAgICAgICB0b3JjaC5GbG9hdFRlbnNvcihudW1weS5hcnJheShzZWxm
LnZpZGVvQXVkaW9bdmlkXSkpLFwKICAgICAgICAgICAgICAgdG9yY2guRmxvYXRUZW5zb3IobnVt
cHkuYXJyYXkoc2VsZi5zcGVha2Vyc1t2aWRdKSksXAogICAgICAgICAgICAgICB0b3JjaC5GbG9h
dFRlbnNvcihudW1weS5hcnJheShbMV0qbGVuKHNlbGYuZW1vdGlvbl9sYWJlbHNbdmlkXSkpKSxc
CiAgICAgICAgICAgICAgIHRvcmNoLkxvbmdUZW5zb3IobnVtcHkuYXJyYXkoc2VsZi5lbW90aW9u
X2xhYmVsc1t2aWRdKSksXAogICAgICAgICAgICAgICB2aWQKCiAgICBkZWYgX19sZW5fXyhzZWxm
KToKICAgICAgICByZXR1cm4gc2VsZi5sZW4KCiAgICBkZWYgY29sbGF0ZV9mbihzZWxmLCBkYXRh
KToKICAgICAgICBkYXQgPSBwZC5EYXRhRnJhbWUoZGF0YSkKICAgICAgICByZXR1cm4gW3BhZF9z
ZXF1ZW5jZShkYXRbaV0pIGlmIGk8NSBlbHNlIHBhZF9zZXF1ZW5jZShkYXRbaV0sIFRydWUpIGlm
IGk8NyBlbHNlIGRhdFtpXS50b2xpc3QoKSBmb3IgaSBpbiBkYXRdCg==
'''),
    'inference.py': ('21f7ce13ac1693d51f605ccc6298f1582432fab27dc3b989cdec0a3ba388e28f', '''
aW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdGltZQppbXBvcnQgdG9y
Y2gKZnJvbSBkYXRhbG9hZGVyIGltcG9ydCBNRUxEUm9iZXJ0YURhdGFzZXQKZnJvbSBtb2RlbCBp
bXBvcnQgRUNFUkMKZnJvbSBza2xlYXJuIGltcG9ydCBtZXRyaWNzCmZyb20gc2tsZWFybi5tZXRy
aWNzIGltcG9ydCBmMV9zY29yZSwgYWNjdXJhY3lfc2NvcmUKZnJvbSB0b3JjaC51dGlscy5kYXRh
IGltcG9ydCBEYXRhTG9hZGVyCmZyb20gbG9zcyBpbXBvcnQgTG9zcwoKaW1wb3J0IHdhcm5pbmdz
Cndhcm5pbmdzLmZpbHRlcndhcm5pbmdzKCJpZ25vcmUiKQoKCmRlZiBnZXRfTUVMRF9iZXJ0X2xv
YWRlcnMocGF0aCwgYmF0Y2hfc2l6ZT0zMiwgY2xhc3NpZnk9J2Vtb3Rpb24nLCBudW1fd29ya2Vy
cz0wLCBwaW5fbWVtb3J5PUZhbHNlKToKICAgIHRyYWluc2V0ID0gTUVMRFJvYmVydGFEYXRhc2V0
KCd0cmFpbicpCiAgICB2YWxpZHNldCA9IE1FTERSb2JlcnRhRGF0YXNldCgndmFsaWQnKQogICAg
dGVzdHNldCA9IE1FTERSb2JlcnRhRGF0YXNldCgndGVzdCcpCgogICAgdHJhaW5fbG9hZGVyID0g
RGF0YUxvYWRlcih0cmFpbnNldCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYmF0Y2hf
c2l6ZT1iYXRjaF9zaXplLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjb2xsYXRlX2Zu
PXRyYWluc2V0LmNvbGxhdGVfZm4sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG51bV93
b3JrZXJzPW51bV93b3JrZXJzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBwaW5fbWVt
b3J5PXBpbl9tZW1vcnksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNodWZmbGU9VHJ1
ZSAgIyBzaHVmZmxlIGZvciB0cmFpbmluZyBkYXRhCiAgICAgICAgICAgICAgICAgICAgICAgICAg
ICAgICkKCiAgICB2YWxpZF9sb2FkZXIgPSBEYXRhTG9hZGVyKHZhbGlkc2V0LAogICAgICAgICAg
ICAgICAgICAgICAgICAgICAgICBiYXRjaF9zaXplPWJhdGNoX3NpemUsCiAgICAgICAgICAgICAg
ICAgICAgICAgICAgICAgIGNvbGxhdGVfZm49dHJhaW5zZXQuY29sbGF0ZV9mbiwKICAgICAgICAg
ICAgICAgICAgICAgICAgICAgICAgbnVtX3dvcmtlcnM9bnVtX3dvcmtlcnMsCiAgICAgICAgICAg
ICAgICAgICAgICAgICAgICAgIHBpbl9tZW1vcnk9cGluX21lbW9yeSkKCiAgICB0ZXN0X2xvYWRl
ciA9IERhdGFMb2FkZXIodGVzdHNldCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBiYXRj
aF9zaXplPWJhdGNoX3NpemUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY29sbGF0ZV9m
bj10ZXN0c2V0LmNvbGxhdGVfZm4sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbnVtX3dv
cmtlcnM9bnVtX3dvcmtlcnMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGluX21lbW9y
eT1waW5fbWVtb3J5KQoKICAgIHJldHVybiB0cmFpbl9sb2FkZXIsIHZhbGlkX2xvYWRlciwgdGVz
dF9sb2FkZXIKCgpkZWYgdHJhaW5fb3JfZXZhbF9tb2RlbChtb2RlbCwgbG9zc19mLCBkYXRhbG9h
ZGVyLCB0cmFpbl9mbGFnPUZhbHNlLCBvcHRpbWl6ZXI9Tm9uZSwgY3VkYV9mbGFnPUZhbHNlLCBm
ZWF0dXJlX3R5cGU9J3RleHQnLCB0YXJnZXRfbmFtZXM9Tm9uZSk6CiAgICBhc3NlcnQgbm90IHRy
YWluX2ZsYWcgb3Igb3B0aW1pemVyICE9IE5vbmUKICAgIGxvc3NlcywgcHJlZHMsIGxhYmVscywg
bWFza3MgPSBbXSwgW10sIFtdLCBbXQoKICAgIGlmIHRyYWluX2ZsYWc6CiAgICAgICAgbW9kZWwu
dHJhaW4oKQogICAgZWxzZToKICAgICAgICBtb2RlbC5ldmFsKCkKCiAgICBmb3IgZGF0YSBpbiBk
YXRhbG9hZGVyOgogICAgICAgIGlmIHRyYWluX2ZsYWc6CiAgICAgICAgICAgIG9wdGltaXplci56
ZXJvX2dyYWQoKQogICAgICAgICMgcm9iZXJ0YV9mZWE6IENMUyBlbWJlZGRpbmcgb2YgbGFzdCBo
aWRkZW4gbGF5ZXIgaW4gUm9CRVJUYQogICAgICAgIGVtb19yb2JlcnRhLCBzZW1fcm9iZXJ0YSwg
YXVkaW8sIHZpc2lvbiwgcW1hc2ssIHVtYXNrLCBsYWJlbCA9IFtkLmN1ZGEoKSBmb3IgZCBpbiBk
YXRhWzotMV1dIGlmIGN1ZGFfZmxhZyBlbHNlIGRhdGFbOi0xXQoKICAgICAgICBzZXFfbGVuZ3Ro
cyA9IFsodW1hc2tbal0gPT0gMSkubm9uemVybygpLnRvbGlzdCgpWy0xXVswXSArIDEgZm9yIGog
aW4gcmFuZ2UobGVuKHVtYXNrKSldCgogICAgICAgIGlmIGFyZ3MuZmVhdHVyZV90eXBlID09ICJt
dWx0aSI6CiAgICAgICAgICAgIGVtb19yb2JlcnRhID0gdG9yY2guY2F0KFtlbW9fcm9iZXJ0YSxh
dWRpbywgdmlzaW9uXSwgZGltPS0xKQogICAgICAgICAgICBzZW1fcm9iZXJ0YSA9IHRvcmNoLmNh
dChbc2VtX3JvYmVydGFdLCBkaW09LTEpCgogICAgICAgIGxvZ19wcm9iID0gbW9kZWwoZW1vX3Jv
YmVydGEsc2VtX3JvYmVydGEsIHFtYXNrLCB1bWFzaywgc2VxX2xlbmd0aHMpCgogICAgICAgIGxh
YmVsID0gdG9yY2guY2F0KFtsYWJlbFtqXVs6c2VxX2xlbmd0aHNbal1dIGZvciBqIGluIHJhbmdl
KGxlbihsYWJlbCkpXSkKICAgICAgICBsb3NzID0gbG9zc19mKGxvZ19wcm9iLCBsYWJlbCkKCiAg
ICAgICAgcHJlZHMuYXBwZW5kKHRvcmNoLmFyZ21heChsb2dfcHJvYiwgMSkuZGF0YS5jcHUoKS5u
dW1weSgpKQogICAgICAgIGxhYmVscy5hcHBlbmQobGFiZWwuZGF0YS5jcHUoKS5udW1weSgpKQoK
ICAgICAgICBsb3NzZXMuYXBwZW5kKGxvc3MuaXRlbSgpKQoKICAgICAgICBpZiB0cmFpbl9mbGFn
OgogICAgICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICAgICAgb3B0aW1pemVyLnN0ZXAo
KQoKICAgIGlmIHByZWRzICE9IFtdOgogICAgICAgIHByZWRzID0gbnAuY29uY2F0ZW5hdGUocHJl
ZHMpCiAgICAgICAgbGFiZWxzID0gbnAuY29uY2F0ZW5hdGUobGFiZWxzKQogICAgZWxzZToKICAg
ICAgICByZXR1cm4gZmxvYXQoJ25hbicpLCBmbG9hdCgnbmFuJyksIGZsb2F0KCduYW4nKSwgW10s
IFtdCgogICAgbGFiZWxzID0gbnAuYXJyYXkobGFiZWxzKQogICAgcHJlZHMgPSBucC5hcnJheShw
cmVkcykKICAgIGF2Z19sb3NzID0gcm91bmQobnAuc3VtKGxvc3NlcykgLyBsZW4obG9zc2VzKSwg
NCkKICAgIGF2Z19hY2N1cmFjeSA9IHJvdW5kKGFjY3VyYWN5X3Njb3JlKGxhYmVscywgcHJlZHMp
ICogMTAwLCAyKQogICAgYXZnX2ZzY29yZSA9IHJvdW5kKGYxX3Njb3JlKGxhYmVscywgcHJlZHMs
IGF2ZXJhZ2U9J3dlaWdodGVkJykgKiAxMDAsIDIpCgogICAgYWxsX21hdHJpeCA9IFtdCiAgICBh
bGxfbWF0cml4LmFwcGVuZCgKICAgICAgICBtZXRyaWNzLmNsYXNzaWZpY2F0aW9uX3JlcG9ydChs
YWJlbHMsIHByZWRzLCB0YXJnZXRfbmFtZXM9dGFyZ2V0X25hbWVzIGlmIHRhcmdldF9uYW1lcyBl
bHNlIE5vbmUsIGRpZ2l0cz00KSkKICAgIGFsbF9tYXRyaXguYXBwZW5kKG1ldHJpY3MuY29uZnVz
aW9uX21hdHJpeChsYWJlbHMsIHByZWRzKSkKCiAgICByZXR1cm4gYXZnX2xvc3MsIGF2Z19hY2N1
cmFjeSwgYXZnX2ZzY29yZSwgYWxsX21hdHJpeCwgW10KCgppZiBfX25hbWVfXyA9PSAnX19tYWlu
X18nOgoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKCiAgICBwYXJzZXIu
YWRkX2FyZ3VtZW50KCctLWZlYXR1cmVfdHlwZScsIHR5cGU9c3RyLCBkZWZhdWx0PSdtdWx0aScs
IGhlbHA9J2ZlYXR1cmUgdHlwZTogdGV4dC9tdWx0aScpCgogICAgcGFyc2VyLmFkZF9hcmd1bWVu
dCgnLS1kYXRhX2RpcicsIHR5cGU9c3RyLCBkZWZhdWx0PScuLi9kYXRhL21lbGQvbWVsZF9mZWF0
dXJlc19yb2JlcnRhLnBrbCcsIGhlbHA9J2RhdGFzZXQgZGlyOiBtZWxkX2ZlYXR1cmVzX3JvYmVy
dGEucGtsJykKCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLWxvYWRfbW9kZWxfc3RhdGVfZGly
JywgdHlwZT1zdHIsIGRlZmF1bHQ9J0VDRVJDX01PREVMLnBrbCcsIGhlbHA9J2xvYWQgbW9kZWwg
c3RhdGUgZGlyJykKCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLWJhc2VfbGF5ZXInLCB0eXBl
PWludCwgZGVmYXVsdD0xLCBoZWxwPSd0aGUgbnVtYmVyIG9mIGJhc2UgbW9kZWwgbGF5ZXJzJykK
CiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLWJhdGNoLXNpemUnLCB0eXBlPWludCwgZGVmYXVs
dD0zMiwgbWV0YXZhcj0nQlMnLCBoZWxwPSdiYXRjaCBzaXplJykKCiAgICBwYXJzZXIuYWRkX2Fy
Z3VtZW50KCctLXVzZV92YWxpZF9mbGFnJywgYWN0aW9uPSdzdG9yZV90cnVlJywgZGVmYXVsdD1U
cnVlLCBoZWxwPSd1c2UgdmFsaWQgc2V0JykKCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLWxy
JywgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjAwMDAxLCBtZXRhdmFyPSdMUicsIGhlbHA9J2xlYXJu
aW5nIHJhdGU6IDAuMDAwNScpCgogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1sMicsIHR5cGU9
ZmxvYXQsIGRlZmF1bHQ9MC4wMDAyLCBtZXRhdmFyPSdMMicsIGhlbHA9J0wyIHJlZ3VsYXJpemF0
aW9uIHdlaWdodCcpCgogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1uby1jdWRhJywgYWN0aW9u
PSdzdG9yZV90cnVlJywgZGVmYXVsdD1GYWxzZSwgaGVscD0nZG9lcyBub3QgdXNlIEdQVScpCgog
ICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1jbGFzc193ZWlnaHQnLCBhY3Rpb249J3N0b3JlX3Ry
dWUnLCBkZWZhdWx0PUZhbHNlLCBoZWxwPSd1c2UgY2xhc3Mgd2VpZ2h0cycpCgogICAgcGFyc2Vy
LmFkZF9hcmd1bWVudCgnLS1jbHNfdHlwZScsIHR5cGU9c3RyLCBkZWZhdWx0PSdlbW90aW9uJywg
aGVscD0nY2hvb3NlIGJldHdlZW4gc2VudGltZW50IG9yIGVtb3Rpb24nKQoKICAgIHBhcnNlci5h
ZGRfYXJndW1lbnQoJy0tRGF0YXNldCcsIHR5cGU9c3RyLCBkZWZhdWx0PSJNRUxEIiwgaGVscD0n
REFUQVNFVCcpCgogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKCiAgICBiYXRjaF9zaXpl
LCBkYXRhX3BhdGgsIGxvYWRfbW9kZWxfc3RhdGVfZGlyLCBiYXNlX2xheWVyLCBmZWF0dXJlX3R5
cGUgPSBcCiAgICAgICAgYXJncy5iYXRjaF9zaXplLCBhcmdzLmRhdGFfZGlyLCBhcmdzLmxvYWRf
bW9kZWxfc3RhdGVfZGlyLCBhcmdzLmJhc2VfbGF5ZXIsIGFyZ3MuZmVhdHVyZV90eXBlCiAgICBj
dWRhX2ZsYWcgPSB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGFuZCBub3QgYXJncy5ub19jdWRh
CgogICAgIyBNRUxEIGRhdGFzZXQKICAgIG5fc3BlYWtlcnMsIGhpZGRlbl9zaXplLCBpbnB1dF9z
aXplID0gOSwgMTI4LCBOb25lCiAgICBpZiBhcmdzLmNsc190eXBlLnN0cmlwKCkubG93ZXIoKSA9
PSAnZW1vdGlvbic6CiAgICAgICAgbl9jbGFzc2VzID0gNwogICAgICAgIHRhcmdldF9uYW1lcyA9
IFsnbmV1JywgJ3N1cicsICdmZWEnLCAnc2FkJywgJ2pveScsICdkaXMnLCAnYW5nJ10KCiAgICAg
ICAgIyBwZXJmb3JtIDEvd2VpZ2h0IGFjY29yZGluZyB0byB0aGUgd2VpZ2h0IG9mIGVhY2ggbGFi
ZWwgaW4gdHJhaW5pbmcgZGF0YQogICAgICAgIGNsYXNzX3dlaWdodHMgPSB0b3JjaC5GbG9hdFRl
bnNvcigKICAgICAgICAgICAgWzEgLyAwLjQ2OTUwNjg1NywgMSAvIDAuMTE5MzQ2MzY3LCAxIC8g
MC4wMjYxMTYxMzcsIDEgLyAwLjA3MzA5NjAwMiwgMSAvIDAuMTY4MzY4ODM2LCAxIC8gMC4wMjYz
MzQ5ODcsCiAgICAgICAgICAgICAxIC8gMC4xMTcyMzA4MTRdKQogICAgICAgIGNsYXNzX3dlaWdo
dHMgPSB0b3JjaC5sb2coY2xhc3Nfd2VpZ2h0cykKICAgIGVsc2U6CiAgICAgICAgbl9jbGFzc2Vz
ID0gMwogICAgICAgIHRhcmdldF9uYW1lcyA9IFsnMCcsICcxJywgJzInXQogICAgICAgIGNsYXNz
X3dlaWdodHMgPSB0b3JjaC5GbG9hdFRlbnNvcihbMS4wLCAxLjAsIDEuMF0pCgogICAgZmVhdDJk
aW0gPSB7J0lTMTAnOiAxNTgyLCAnM0RDTk4nOiA1MTIsICd0ZXh0Q05OJzogMTAwLCAnYmVydCc6
IDc2OCwgJ2RlbnNlZmFjZSc6IDM0MiwgJ01FTERfdGV4dCc6IDYwMCwKICAgICAgICAgICAgICAg
ICdNRUxEX2F1ZGlvJzogMzAwfQogICAgRF9hdWRpbyA9IGZlYXQyZGltWydJUzEwJ10gaWYgYXJn
cy5EYXRhc2V0ID09ICdJRU1PQ0FQJyBlbHNlIGZlYXQyZGltWydNRUxEX2F1ZGlvJ10KICAgIERf
dmlzdWFsID0gZmVhdDJkaW1bJ2RlbnNlZmFjZSddCiAgICBEX3RleHQgPSAxMDI0CgogICAgaWYg
ZmVhdHVyZV90eXBlID09ICd0ZXh0JzoKICAgICAgICBpbnB1dF9zaXplID0gRF90ZXh0CiAgICBl
bGlmIGZlYXR1cmVfdHlwZSA9PSAibXVsdGkiOgogICAgICAgIGlucHV0X3NpemUgPSBEX3RleHQg
KyBEX2F1ZGlvICsgRF92aXN1YWwKICAgIGVsc2U6CiAgICAgICAgcHJpbnQoJ0Vycm9yOiBmZWF0
dXJlX3R5cGUgbm90IHNldC4nKQogICAgICAgIGV4aXQoMCkKCiAgICBtb2RlbCA9IEVDRVJDKGFy
Z3MsIGRfdD1EX3RleHQsIGRfYT1EX2F1ZGlvLCBkX3Y9RF92aXN1YWwsCiAgICAgICAgICAgICAg
ICAgIGJhc2VfbGF5ZXI9YmFzZV9sYXllciwKICAgICAgICAgICAgICAgICAgaW5wdXRfc2l6ZT1p
bnB1dF9zaXplLAogICAgICAgICAgICAgICAgICBoaWRkZW5fc2l6ZT1oaWRkZW5fc2l6ZSwKICAg
ICAgICAgICAgICAgICAgbl9zcGVha2Vycz1uX3NwZWFrZXJzLAogICAgICAgICAgICAgICAgICBu
X2NsYXNzZXM9bl9jbGFzc2VzLAogICAgICAgICAgICAgICAgICBjdWRhX2ZsYWc9Y3VkYV9mbGFn
KQogICAgaWYgY3VkYV9mbGFnOgogICAgICAgIHByaW50KCdSdW5uaW5nIG9uIEdQVScpCiAgICAg
ICAgY2xhc3Nfd2VpZ2h0cyA9IGNsYXNzX3dlaWdodHMuY3VkYSgpCiAgICAgICAgbW9kZWwuY3Vk
YSgpCiAgICBlbHNlOgogICAgICAgIHByaW50KCdSdW5uaW5nIG9uIENQVScpCgogICAgbmFtZSA9
ICdFQ0VSQycKICAgIHByaW50KCJUaGUgbW9kZWwgaGF2ZSB7fSBwYXJhbWVydGVycyBpbiB0b3Rh
bCIuZm9ybWF0KHN1bSh4Lm51bWVsKCkgZm9yIHggaW4gbW9kZWwucGFyYW1ldGVycygpKSkpCiAg
ICBwcmludCgnUnVubmluZyBvbiB0aGUge30gZmVhdHVyZXMuLi4uLi4uLicuZm9ybWF0KGZlYXR1
cmVfdHlwZSkpCgogICAgdHJhaW5fbG9hZGVyLCB2YWxpZF9sb2FkZXIsIHRlc3RfbG9hZGVyID0g
Z2V0X01FTERfYmVydF9sb2FkZXJzKGRhdGFfcGF0aCwgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLAog
ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg
ICAgICAgICAgIGNsYXNzaWZ5PWFyZ3MuY2xzX3R5cGUsIG51bV93b3JrZXJzPTApCiAgICBsb3Nz
X2YgPSBMb3NzKGFscGhhPWNsYXNzX3dlaWdodHMgaWYgYXJncy5jbGFzc193ZWlnaHQgZWxzZSBO
b25lKQoKICAgIHN0YXJ0X3RpbWUgPSB0aW1lLnRpbWUoKQogICAgbW9kZWwubG9hZF9zdGF0ZV9k
aWN0KHRvcmNoLmxvYWQoYXJncy5sb2FkX21vZGVsX3N0YXRlX2RpcikpCiAgICB0ZXN0X2xvc3Ms
IHRlc3RfYWNjLCB0ZXN0X2ZzY29yZSwgdGVzdF9tZXRyaWNzLCB0ZXN0X291dHB1dHMgPSB0cmFp
bl9vcl9ldmFsX21vZGVsKG1vZGVsPW1vZGVsLCBsb3NzX2Y9bG9zc19mLCBkYXRhbG9hZGVyPXRl
c3RfbG9hZGVyLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg
ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjdWRhX2ZsYWc9Y3VkYV9m
bGFnLCBmZWF0dXJlX3R5cGU9ZmVhdHVyZV90eXBlLAogICAgICAgICAgICAgICAgICAgICAgICAg
ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg
ICAgICB0YXJnZXRfbmFtZXM9dGFyZ2V0X25hbWVzKQoKICAgIHByaW50KCd0ZXN0X2xvc3M6IHt9
LCB0ZXN0X2FjYzoge30sIHRlc3RfZnNjb3JlOiB7fSwgdGltZToge30gc2VjJy5mb3JtYXQodGVz
dF9sb3NzLCB0ZXN0X2FjYywgdGVzdF9mc2NvcmUsIHJvdW5kKHRpbWUudGltZSgpIC0gc3RhcnRf
dGltZSwgMikpKQogICAgcHJpbnQodGVzdF9tZXRyaWNzWzBdKQogICAgcHJpbnQodGVzdF9tZXRy
aWNzWzFdKQo=
'''),
    'loss.py': ('c348375c27c366e11f2509de30dd0c5c0f224adcf0eb434bc0d0d9fa12045246', '''
aW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgpmcm9tIHRvcmNoLmF1dG9ncmFkIGlt
cG9ydCBWYXJpYWJsZQoKCmNsYXNzIExvc3Mobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhz
ZWxmLCBnYW1tYT0xLCBhbHBoYT1Ob25lLCBzaXplX2F2ZXJhZ2U9VHJ1ZSk6CiAgICAgICAgc3Vw
ZXIoTG9zcywgc2VsZikuX19pbml0X18oKQogICAgICAgIHNlbGYuZ2FtbWEgPSBnYW1tYQogICAg
ICAgIHNlbGYuYWxwaGEgPSBhbHBoYQogICAgICAgIGlmIGlzaW5zdGFuY2UoYWxwaGEsIChmbG9h
dCwgaW50KSk6IHNlbGYuYWxwaGEgPSB0b3JjaC5UZW5zb3IoW2FscGhhLCAxIC0gYWxwaGFdKQog
ICAgICAgIGlmIGlzaW5zdGFuY2UoYWxwaGEsIGxpc3QpOiBzZWxmLmFscGhhID0gdG9yY2guVGVu
c29yKGFscGhhKQogICAgICAgIHNlbGYuc2l6ZV9hdmVyYWdlID0gc2l6ZV9hdmVyYWdlCgogICAg
ZGVmIGZvcndhcmQoc2VsZiwgaW5wdXQsIHRhcmdldCk6CiAgICAgICAgaWYgaW5wdXQuZGltKCkg
PiAyOgogICAgICAgICAgICBpbnB1dCA9IGlucHV0LnZpZXcoaW5wdXQuc2l6ZSgwKSwgaW5wdXQu
c2l6ZSgxKSwgLTEpCiAgICAgICAgICAgIGlucHV0ID0gaW5wdXQudHJhbnNwb3NlKDEsIDIpCiAg
ICAgICAgICAgIGlucHV0ID0gaW5wdXQuY29udGlndW91cygpLnZpZXcoLTEsIGlucHV0LnNpemUo
MikpCiAgICAgICAgdGFyZ2V0ID0gdGFyZ2V0LnZpZXcoLTEsIDEpCgogICAgICAgIGxvZ3B0ID0g
aW5wdXQuZ2F0aGVyKDEsIHRhcmdldCkudmlldygtMSkKICAgICAgICBwdCA9IFZhcmlhYmxlKGxv
Z3B0LmRhdGEuZXhwKCkpCgogICAgICAgIGlmIHNlbGYuYWxwaGEgaXMgbm90IE5vbmU6CiAgICAg
ICAgICAgIGlmIHNlbGYuYWxwaGEudHlwZSgpICE9IGlucHV0LmRhdGEudHlwZSgpOgogICAgICAg
ICAgICAgICAgc2VsZi5hbHBoYSA9IHNlbGYuYWxwaGEudHlwZV9hcyhpbnB1dC5kYXRhKQogICAg
ICAgICAgICBhdCA9IHNlbGYuYWxwaGEuZ2F0aGVyKDAsIHRhcmdldC5kYXRhLnZpZXcoLTEpKQog
ICAgICAgICAgICBsb2dwdCA9IGxvZ3B0ICogVmFyaWFibGUoYXQpCgogICAgICAgIGxvc3MgPSAt
MSAqICgxIC0gcHQpICoqIHNlbGYuZ2FtbWEgKiBsb2dwdAogICAgICAgIGlmIHNlbGYuc2l6ZV9h
dmVyYWdlOgogICAgICAgICAgICByZXR1cm4gbG9zcy5tZWFuKCkKICAgICAgICBlbHNlOgogICAg
ICAgICAgICByZXR1cm4gbG9zcy5zdW0oKQo=
'''),
    'l2_posthoc.py': ('...
fSBNQiknKQogICAgcmV0dXJuIHpwYXRoCg==
'''),
    'l2_expert.py': ('...
TUIpJykKICAgIHJldHVybiB6cGF0aAo=
'''),
    'l2_llm.py': ('...
Oi4xZn0gTUIpJykKICAgIHJldHVybiB6cGF0aAo=
'''),
    'l2_llm_worker.py': ('154187d9700acdf300f8ea33398f229306621ef446502393d199a0b6c7169635', '''
IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKV29ya2VyIGZvciBFeHBlcmltZW50IDUgKExMTSBl
eHBlcnQpLiBSdW5zIGluIGl0cyBvd24gcHJvY2Vzcywgb25lIHBlciBHUFUsIHNvIHNlZWRzIGNh
biB0cmFpbiBpbiBwYXJhbGxlbC4KCiAgICBweXRob24gbDJfbGxtX3dvcmtlci5weSB0cmFpbiAg
IC0tY2ZnIHJ1bl9jZmcuanNvbiAtLXNlZWQgMSAtLW91dCBESVIKICAgIHB5dGhvbiBsMl9sbG1f
d29ya2VyLnB5IHByZWRpY3QgLS1jZmcgcnVuX2NmZy5qc29uIC0tYWRhcHRlciBESVIvYWRhcHRl
ciAtLWlucHV0cyBpbnB1dHMuanNvbiAtLW91dCBGSUxFLm5wegoKdHJhaW46IGZpbmUtdHVuZXMg
YSBMb1JBIGFkYXB0ZXIgcGx1cyB0aGUgY2xhc3NpZmljYXRpb24gaGVhZCBvbiB0aGUgdHJhaW5p
bmcgaW5wdXRzLCBldmFsdWF0ZXMgb24gZGV2IGV2ZXJ5CmNmZ1snZXZhbF9ldmVyeSddIGVwb2No
cyBhbmQga2VlcHMgdGhlIGFkYXB0ZXIgd2l0aCB0aGUgYmVzdCBkZXYgbWFjcm8tRjEgKERJUi9h
ZGFwdGVyLCBESVIvZGV2X2xvZ3AubnB6LApESVIvcmVzdWx0Lmpzb24pLiBJdCBuZXZlciByZWFk
cyB0ZXN0IGlucHV0cy4gcHJlZGljdDogd3JpdGVzIGxvZy1wcm9iYWJpbGl0aWVzIGZvciB0aGUg
Z2l2ZW4gaW5wdXRzIChubyBsYWJlbHMpLgoiIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29u
CmltcG9ydCBtYXRoCmltcG9ydCBvcwppbXBvcnQgcmFuZG9tCmltcG9ydCB0aW1lCgppbXBvcnQg
bnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYK
CkMgPSA3CgoKZGVmIGxvZygqYSk6CiAgICBwcmludCgqYSwgZmx1c2g9VHJ1ZSkKCgpkZWYgbWFj
cm9fZjEoeSwgcCk6CiAgICBjbSA9IG5wLmJpbmNvdW50KHkgKiBDICsgcCwgbWlubGVuZ3RoPUMg
KiBDKS5yZXNoYXBlKEMsIEMpCiAgICB0cCwgcywgcSA9IG5wLmRpYWcoY20pLmFzdHlwZShmbG9h
dCksIGNtLnN1bSgxKSwgY20uc3VtKDApCiAgICBmMSA9IG5wLmRpdmlkZSgyICogdHAsIHMgKyBx
LCBvdXQ9bnAuemVyb3MoQyksIHdoZXJlPShzICsgcSkgPiAwKQogICAgcmV0dXJuIDEwMCAqIGYx
Lm1lYW4oKSwgMTAwICogZjEKCgpkZWYgbG9hZF9tb2RlbChjZmcsIGFkYXB0ZXI9Tm9uZSk6CiAg
ICBmcm9tIHBlZnQgaW1wb3J0IExvcmFDb25maWcsIFBlZnRNb2RlbCwgZ2V0X3BlZnRfbW9kZWws
IHByZXBhcmVfbW9kZWxfZm9yX2tiaXRfdHJhaW5pbmcKICAgIGZyb20gdHJhbnNmb3JtZXJzIGlt
cG9ydCBBdXRvTW9kZWxGb3JTZXF1ZW5jZUNsYXNzaWZpY2F0aW9uLCBCaXRzQW5kQnl0ZXNDb25m
aWcKICAgIGN1ZGEgPSB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpCiAgICBxdWFudCA9IGJvb2wo
Y2ZnWydxdWFudGl6ZSddKSBhbmQgY3VkYQogICAga3cgPSBkaWN0KG51bV9sYWJlbHM9QykKICAg
IGlmIHF1YW50OgogICAgICAgIGt3WydxdWFudGl6YXRpb25fY29uZmlnJ10gPSBCaXRzQW5kQnl0
ZXNDb25maWcobG9hZF9pbl80Yml0PVRydWUsIGJuYl80Yml0X3F1YW50X3R5cGU9J25mNCcsCiAg
ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYm5i
XzRiaXRfdXNlX2RvdWJsZV9xdWFudD1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAg
ICAgICAgICAgICAgICAgICAgICAgICAgICAgIGJuYl80Yml0X2NvbXB1dGVfZHR5cGU9dG9yY2gu
ZmxvYXQxNikKICAgICAgICBrd1snZGV2aWNlX21hcCddID0geycnOiAwfQogICAgZHR5cGUgPSB0
b3JjaC5mbG9hdDE2IGlmIGN1ZGEgZWxzZSB0b3JjaC5mbG9hdDMyCiAgICB0cnk6CiAgICAgICAg
bW9kZWwgPSBBdXRvTW9kZWxGb3JTZXF1ZW5jZUNsYXNzaWZpY2F0aW9uLmZyb21fcHJldHJhaW5l
ZChjZmdbJ21vZGVsJ10sIGR0eXBlPWR0eXBlLCAqKmt3KQogICAgZXhjZXB0IFR5cGVFcnJvcjog
ICMgb2xkZXIgdHJhbnNmb3JtZXJzCiAgICAgICAgbW9kZWwgPSBBdXRvTW9kZWxGb3JTZXF1ZW5j
ZUNsYXNzaWZpY2F0aW9uLmZyb21fcHJldHJhaW5lZChjZmdbJ21vZGVsJ10sIHRvcmNoX2R0eXBl
PWR0eXBlLCAqKmt3KQogICAgbW9kZWwuY29uZmlnLnBhZF90b2tlbl9pZCA9IGNmZ1sncGFkX2lk
J10KICAgIG1vZGVsLmNvbmZpZy51c2VfY2FjaGUgPSBGYWxzZQogICAgaWYgYWRhcHRlciBpcyBO
b25lOgogICAgICAgIGlmIHF1YW50OgogICAgICAgICAgICBtb2RlbCA9IHByZXBhcmVfbW9kZWxf
Zm9yX2tiaXRfdHJhaW5pbmcobW9kZWwsIHVzZV9ncmFkaWVudF9jaGVja3BvaW50aW5nPVRydWUs
CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBncmFk
aWVudF9jaGVja3BvaW50aW5nX2t3YXJncz17J3VzZV9yZWVudHJhbnQnOiBGYWxzZX0pCiAgICAg
ICAgZWxpZiBjdWRhOgogICAgICAgICAgICBtb2RlbC5ncmFkaWVudF9jaGVja3BvaW50aW5nX2Vu
YWJsZShncmFkaWVudF9jaGVja3BvaW50aW5nX2t3YXJncz17J3VzZV9yZWVudHJhbnQnOiBGYWxz
ZX0pCiAgICAgICAgICAgIG1vZGVsLmVuYWJsZV9pbnB1dF9yZXF1aXJlX2dyYWRzKCkKICAgICAg
ICBtb2RlbCA9IGdldF9wZWZ0X21vZGVsKG1vZGVsLCBMb3JhQ29uZmlnKHI9Y2ZnWydsb3JhX3In
XSwgbG9yYV9hbHBoYT1jZmdbJ2xvcmFfYWxwaGEnXSwKICAgICAgICAgICAgICAgICAgICAgICAg
ICAgICAgICAgICAgICAgICAgICAgICAgIGxvcmFfZHJvcG91dD1jZmdbJ2xvcmFfZHJvcG91dCdd
LCB0YXNrX3R5cGU9J1NFUV9DTFMnLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg
ICAgICAgICAgICAgICAgdGFyZ2V0X21vZHVsZXM9Y2ZnWydsb3JhX3RhcmdldHMnXSkpCiAgICBl
bHNlOgogICAgICAgIG1vZGVsID0gUGVmdE1vZGVsLmZyb21fcHJldHJhaW5lZChtb2RlbCwgYWRh
cHRlcikKICAgIGlmIG5vdCBxdWFudDoKICAgICAgICBtb2RlbC50bygnY3VkYScgaWYgY3VkYSBl
bHNlICdjcHUnKQogICAgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpOiAgIyB0cmFpbmFibGUg
d2VpZ2h0cyBpbiBmcDMyIChuZWVkZWQgYnkgdGhlIGZwMTYgZ3JhZGllbnQgc2NhbGVyKQogICAg
ICAgIGlmIHAucmVxdWlyZXNfZ3JhZCBhbmQgcC5kdHlwZSAhPSB0b3JjaC5mbG9hdDMyOgogICAg
ICAgICAgICBwLmRhdGEgPSBwLmRhdGEuZmxvYXQoKQogICAgcmV0dXJuIG1vZGVsCgoKZGVmIGNv
bGxhdGUoaWRzLCBpZHgsIHBhZF9pZCwgZGV2aWNlKToKICAgIEwgPSBtYXgobGVuKGlkc1tpXSkg
Zm9yIGkgaW4gaWR4KQogICAgeCA9IHRvcmNoLmZ1bGwoKGxlbihpZHgpLCBMKSwgcGFkX2lkLCBk
dHlwZT10b3JjaC5sb25nKQogICAgYSA9IHRvcmNoLnplcm9zKChsZW4oaWR4KSwgTCksIGR0eXBl
PXRvcmNoLmxvbmcpCiAgICBmb3IgciwgaSBpbiBlbnVtZXJhdGUoaWR4KTogICMgcmlnaHQgcGFk
ZGluZzogdGhlIGNsYXNzaWZpZXIgcmVhZHMgdGhlIGxhc3QgcmVhbCB0b2tlbgogICAgICAgIHhb
ciwgOmxlbihpZHNbaV0pXSA9IHRvcmNoLnRlbnNvcihpZHNbaV0pCiAgICAgICAgYVtyLCA6bGVu
KGlkc1tpXSldID0gMQogICAgcmV0dXJuIHgudG8oZGV2aWNlKSwgYS50byhkZXZpY2UpCgoKZGVm
IHByZWRpY3QobW9kZWwsIGlkcywgcGFkX2lkLCBicywgZGV2aWNlKToKICAgIG1vZGVsLmV2YWwo
KQogICAgb3JkZXIgPSBucC5hcmdzb3J0KFtsZW4ocykgZm9yIHMgaW4gaWRzXSkKICAgIG91dCA9
IG5wLnplcm9zKChsZW4oaWRzKSwgQykpCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAg
ICBmb3IgYiBpbiByYW5nZSgwLCBsZW4oaWRzKSwgYnMpOgogICAgICAgICAgICBpZHggPSBvcmRl
cltiOmIgKyBic10KICAgICAgICAgICAgeCwgYSA9IGNvbGxhdGUoaWRzLCBpZHgsIHBhZF9pZCwg
ZGV2aWNlKQogICAgICAgICAgICB3aXRoIHRvcmNoLmF1dG9jYXN0KGRldmljZV90eXBlPSdjdWRh
JywgZHR5cGU9dG9yY2guZmxvYXQxNiwgZW5hYmxlZD1kZXZpY2UgPT0gJ2N1ZGEnKToKICAgICAg
ICAgICAgICAgIGxvZ2l0cyA9IG1vZGVsKGlucHV0X2lkcz14LCBhdHRlbnRpb25fbWFzaz1hKS5s
b2dpdHMKICAgICAgICAgICAgb3V0W2lkeF0gPSBGLmxvZ19zb2Z0bWF4KGxvZ2l0cy5mbG9hdCgp
LCBkaW09LTEpLmNwdSgpLm51bXB5KCkKICAgIGlmIG5vdCBucC5pc2Zpbml0ZShvdXQpLmFsbCgp
OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignbm9uLWZpbml0ZSBwcmVkaWN0aW9ucyAoZnAx
NiBvdmVyZmxvdz8pJykKICAgIHJldHVybiBvdXQKCgpkZWYgdHJhaW4oY2ZnLCBzZWVkLCBvdXQp
OgogICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IGdldF9jb3NpbmVfc2NoZWR1bGVfd2l0aF93
YXJtdXAKICAgIHJhbmRvbS5zZWVkKHNlZWQpCiAgICBucC5yYW5kb20uc2VlZChzZWVkKQogICAg
dG9yY2gubWFudWFsX3NlZWQoc2VlZCkKICAgIHRvcmNoLmN1ZGEubWFudWFsX3NlZWRfYWxsKHNl
ZWQpCiAgICB3aXRoIG9wZW4oY2ZnWyd0cmFpbnZhbF9wYXRoJ10pIGFzIGY6CiAgICAgICAgZGF0
YSA9IGpzb24ubG9hZChmKQogICAgdHJfaWRzLCB0cl95ID0gZGF0YVsndHJhaW4nXVsnaWRzJ10s
IG5wLmFzYXJyYXkoZGF0YVsndHJhaW4nXVsneSddKQogICAgZHZfaWRzLCBkdl95ID0gZGF0YVsn
dmFsaWQnXVsnaWRzJ10sIG5wLmFzYXJyYXkoZGF0YVsndmFsaWQnXVsneSddKQogICAgZGV2aWNl
ID0gJ2N1ZGEnIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAnY3B1JwogICAgdDAg
PSB0aW1lLnRpbWUoKQogICAgbW9kZWwgPSBsb2FkX21vZGVsKGNmZykKICAgIGxvZyhmJ1tzZWVk
IHtzZWVkfV0gbW9kZWwgbG9hZGVkIG9uIHtkZXZpY2V9IGluIHt0aW1lLnRpbWUoKSAtIHQwOi4w
Zn1zOyB0cmFpbmFibGUgcGFyYW1ldGVycyAnCiAgICAgICAgZid7c3VtKHAubnVtZWwoKSBmb3Ig
cCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkgaWYgcC5yZXF1aXJlc19ncmFkKSAvIDFlNjouMWZ9TScp
CiAgICBwYXJhbXMgPSBbcCBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkgaWYgcC5yZXF1aXJl
c19ncmFkXQogICAgb3B0ID0gdG9yY2gub3B0aW0uQWRhbVcocGFyYW1zLCBscj1jZmdbJ2xyJ10s
IHdlaWdodF9kZWNheT0wLjApCiAgICBtaWNybywgYWNjdW0gPSBjZmdbJ21pY3JvX2JhdGNoJ10s
IGNmZ1snZ3JhZF9hY2N1bSddCiAgICBuID0gbGVuKHRyX2lkcykKICAgIG1pY3JvX3Blcl9lcG9j
aCA9IG1hdGguY2VpbChuIC8gbWljcm8pCiAgICBvcHRfc3RlcHMgPSBtYXRoLmNlaWwobWljcm9f
cGVyX2Vwb2NoIC8gYWNjdW0pICogY2ZnWydlcG9jaHMnXQogICAgc2NoZWQgPSBnZXRfY29zaW5l
X3NjaGVkdWxlX3dpdGhfd2FybXVwKG9wdCwgaW50KGNmZ1snd2FybXVwJ10gKiBvcHRfc3RlcHMp
LCBvcHRfc3RlcHMpCiAgICBzY2FsZXIgPSB0b3JjaC5hbXAuR3JhZFNjYWxlcignY3VkYScsIGVu
YWJsZWQ9ZGV2aWNlID09ICdjdWRhJykKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhz
ZWVkKQogICAgZXZhbF9ldmVyeSA9IG1heCgxLCBpbnQocm91bmQobWljcm9fcGVyX2Vwb2NoICog
Y2ZnWydldmFsX2V2ZXJ5J10pKSkKICAgIGxlbmd0aHMgPSBucC5hc2FycmF5KFtsZW4ocykgZm9y
IHMgaW4gdHJfaWRzXSkKICAgIGJlc3QsIGJhZCwgY3VydmUsIG1zdGVwLCBzdG9wLCB0MCwgYmFk
X2xvc3MgPSAtMS4wLCAwLCBbXSwgMCwgRmFsc2UsIHRpbWUudGltZSgpLCAwCiAgICBmb3IgZXBv
Y2ggaW4gcmFuZ2UoY2ZnWydlcG9jaHMnXSk6CiAgICAgICAgbW9kZWwudHJhaW4oKQogICAgICAg
IHBlcm0sIG1lZ2EgPSBybmcucGVybXV0YXRpb24obiksIG1pY3JvICogNjQKICAgICAgICBiYXRj
aGVzID0gW10KICAgICAgICBmb3IgaSBpbiByYW5nZSgwLCBuLCBtZWdhKTogICMgZ3JvdXAgc2lt
aWxhciBsZW5ndGhzIHRvIHNhdmUgcGFkZGluZywgdGhlbiBzaHVmZmxlIHRoZSBiYXRjaGVzCiAg
ICAgICAgICAgIGNoID0gcGVybVtpOmkgKyBtZWdhXQogICAgICAgICAgICBjaCA9IGNoW25wLmFy
Z3NvcnQobGVuZ3Roc1tjaF0sIGtpbmQ9J3N0YWJsZScpXQogICAgICAgICAgICBiYXRjaGVzICs9
IFtjaFtqOmogKyBtaWNyb10gZm9yIGogaW4gcmFuZ2UoMCwgbGVuKGNoKSwgbWljcm8pXQogICAg
ICAgIG9yZGVyID0gcm5nLnBlcm11dGF0aW9uKGxlbihiYXRjaGVzKSkKICAgICAgICBmb3Igaywg
YmkgaW4gZW51bWVyYXRlKG9yZGVyKToKICAgICAgICAgICAgaWR4ID0gYmF0Y2hlc1tiaV0KICAg
ICAgICAgICAgeCwgYSA9IGNvbGxhdGUodHJfaWRzLCBpZHgsIGNmZ1sncGFkX2lkJ10sIGRldmlj
ZSkKICAgICAgICAgICAgeSA9IHRvcmNoLmFzX3RlbnNvcih0cl95W2lkeF0sIGRldmljZT1kZXZp
Y2UpCiAgICAgICAgICAgIHdpdGggdG9yY2guYXV0b2Nhc3QoZGV2aWNlX3R5cGU9J2N1ZGEnLCBk
dHlwZT10b3JjaC5mbG9hdDE2LCBlbmFibGVkPWRldmljZSA9PSAnY3VkYScpOgogICAgICAgICAg
ICAgICAgbG9naXRzID0gbW9kZWwoaW5wdXRfaWRzPXgsIGF0dGVudGlvbl9tYXNrPWEpLmxvZ2l0
cwogICAgICAgICAgICBsb3NzID0gRi5jcm9zc19lbnRyb3B5KGxvZ2l0cy5mbG9hdCgpLCB5KSAv
IGFjY3VtCiAgICAgICAgICAgIGlmIG5vdCB0b3JjaC5pc2Zpbml0ZShsb3NzKTogICMgZnAxNiB0
cm91YmxlOiBza2lwIHRoZSBiYXRjaCwgZ2l2ZSB1cCBlYXJseSBpZiBpdCBrZWVwcyBoYXBwZW5p
bmcKICAgICAgICAgICAgICAgIGJhZF9sb3NzICs9IDEKICAgICAgICAgICAgICAgIG9wdC56ZXJv
X2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgICAgICAgICAgICAgIGlmIGJhZF9sb3NzID4gMjA6
CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCd0cmFpbmluZyBkaXZlcmdl
ZDogbW9yZSB0aGFuIDIwIG5vbi1maW5pdGUgbG9zc2VzJykKICAgICAgICAgICAgICAgIGNvbnRp
bnVlCiAgICAgICAgICAgIHNjYWxlci5zY2FsZShsb3NzKS5iYWNrd2FyZCgpCiAgICAgICAgICAg
IG1zdGVwICs9IDEKICAgICAgICAgICAgbGFzdCA9IGsgPT0gbGVuKG9yZGVyKSAtIDEKICAgICAg
ICAgICAgaWYgbXN0ZXAgJSBhY2N1bSA9PSAwIG9yIGxhc3Q6CiAgICAgICAgICAgICAgICBzY2Fs
ZXIudW5zY2FsZV8ob3B0KQogICAgICAgICAgICAgICAgdG9yY2gubm4udXRpbHMuY2xpcF9ncmFk
X25vcm1fKHBhcmFtcywgMS4wKQogICAgICAgICAgICAgICAgc2NhbGVyLnN0ZXAob3B0KQogICAg
ICAgICAgICAgICAgc2NhbGVyLnVwZGF0ZSgpCiAgICAgICAgICAgICAgICBvcHQuemVyb19ncmFk
KHNldF90b19ub25lPVRydWUpCiAgICAgICAgICAgICAgICBzY2hlZC5zdGVwKCkKICAgICAgICAg
ICAgdGltZV91cCA9ICh0aW1lLnRpbWUoKSAtIHQwKSAvIDYwID4gY2ZnLmdldCgnbWF4X21pbnV0
ZXMnLCBmbG9hdCgnaW5mJykpCiAgICAgICAgICAgIGlmIG1zdGVwICUgZXZhbF9ldmVyeSA9PSAw
IG9yIGxhc3Qgb3IgdGltZV91cDoKICAgICAgICAgICAgICAgIGxwID0gcHJlZGljdChtb2RlbCwg
ZHZfaWRzLCBjZmdbJ3BhZF9pZCddLCBjZmdbJ2V2YWxfYmF0Y2gnXSwgZGV2aWNlKQogICAgICAg
ICAgICAgICAgbWYsIGYxID0gbWFjcm9fZjEoZHZfeSwgbHAuYXJnbWF4KDEpKQogICAgICAgICAg
ICAgICAgYWNjID0gMTAwICogZmxvYXQoKGxwLmFyZ21heCgxKSA9PSBkdl95KS5tZWFuKCkpCiAg
ICAgICAgICAgICAgICBjdXJ2ZS5hcHBlbmQoZGljdChlcG9jaD1lcG9jaCwgbWljcm9fc3RlcD1t
c3RlcCwgZGV2X21hY3JvX2YxPW1mLCBkZXZfYWNjPWFjYywgZjE9ZjEudG9saXN0KCksCiAgICAg
ICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtaW51dGVzPXJvdW5kKCh0aW1lLnRpbWUoKSAt
IHQwKSAvIDYwLCAxKSkpCiAgICAgICAgICAgICAgICBsb2coZidbc2VlZCB7c2VlZH1dIGVwb2No
IHtlcG9jaH0gc3RlcCB7bXN0ZXB9L3ttaWNyb19wZXJfZXBvY2ggKiBjZmdbImVwb2NocyJdfTog
ZGV2IG1hY3JvLUYxIHttZjouMmZ9LCAnCiAgICAgICAgICAgICAgICAgICAgZidhY2Mge2FjYzou
MmZ9LCBGZWFyIHtmMVsyXTouMWZ9LCBEaXNndXN0IHtmMVs1XTouMWZ9LCBsb3NzIHtsb3NzLml0
ZW0oKSAqIGFjY3VtOi4zZn0gJwogICAgICAgICAgICAgICAgICAgIGYnKHtjdXJ2ZVstMV1bIm1p
bnV0ZXMiXX0gbWluKScpCiAgICAgICAgICAgICAgICBpZiBtZiA+IGJlc3QgKyAxZS05OgogICAg
ICAgICAgICAgICAgICAgIGJlc3QsIGJhZCA9IG1mLCAwCiAgICAgICAgICAgICAgICAgICAgbW9k
ZWwuc2F2ZV9wcmV0cmFpbmVkKG9zLnBhdGguam9pbihvdXQsICdhZGFwdGVyJykpCiAgICAgICAg
ICAgICAgICAgICAgbnAuc2F2ZXpfY29tcHJlc3NlZChvcy5wYXRoLmpvaW4ob3V0LCAnZGV2X2xv
Z3AubnB6JyksIGxvZ3A9bHAuYXN0eXBlKG5wLmZsb2F0MzIpLCB5PWR2X3kpCiAgICAgICAgICAg
ICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIGJhZCArPSAxCiAgICAgICAgICAgICAgICBp
ZiBiYWQgPj0gY2ZnWydwYXRpZW5jZSddIG9yIHRpbWVfdXA6ICAjIHBhdGllbmNlIGV4aGF1c3Rl
ZCwgb3IgdGhlIHRyYWluaW5nIHRpbWUgY2FwIHJlYWNoZWQKICAgICAgICAgICAgICAgICAgICBz
dG9wID0gVHJ1ZQogICAgICAgICAgICAgICAgICAgIGlmIHRpbWVfdXA6CiAgICAgICAgICAgICAg
ICAgICAgICAgIGxvZyhmJ1tzZWVkIHtzZWVkfV0gdHJhaW5pbmcgdGltZSBjYXAgb2Yge2NmZ1si
bWF4X21pbnV0ZXMiXX0gbWluIHJlYWNoZWQ7IHN0b3BwaW5nJykKICAgICAgICAgICAgICAgICAg
ICBicmVhawogICAgICAgICAgICAgICAgbW9kZWwudHJhaW4oKQogICAgICAgIGlmIHN0b3A6CiAg
ICAgICAgICAgIGJyZWFrCiAgICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKG91dCwgJ3Jlc3VsdC5q
c29uJyksICd3JykgYXMgZjoKICAgICAgICBqc29uLmR1bXAoZGljdChzZWVkPXNlZWQsIGJlc3Rf
ZGV2X21hY3JvX2YxPWJlc3QsIGN1cnZlPWN1cnZlLCBtaW51dGVzPXJvdW5kKCh0aW1lLnRpbWUo
KSAtIHQwKSAvIDYwLCAxKSwKICAgICAgICAgICAgICAgICAgICAgICBzdG9wcGVkX2Vhcmx5PXN0
b3ApLCBmLCBpbmRlbnQ9MikKICAgIGxvZyhmJ1tzZWVkIHtzZWVkfV0gZG9uZTogYmVzdCBkZXYg
bWFjcm8tRjEge2Jlc3Q6LjJmfScpCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJn
dW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCdtb2RlJywgY2hvaWNlcz1bJ3RyYWlu
JywgJ3ByZWRpY3QnXSkKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1jZmcnLCByZXF1aXJlZD1UcnVl
KQogICAgYXAuYWRkX2FyZ3VtZW50KCctLXNlZWQnLCB0eXBlPWludCwgZGVmYXVsdD0wKQogICAg
YXAuYWRkX2FyZ3VtZW50KCctLW91dCcsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1l
bnQoJy0tYWRhcHRlcicpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0taW5wdXRzJykKICAgIGEgPSBh
cC5wYXJzZV9hcmdzKCkKICAgIHdpdGggb3BlbihhLmNmZykgYXMgZjoKICAgICAgICBjZmcgPSBq
c29uLmxvYWQoZikKICAgIGlmIGEubW9kZSA9PSAndHJhaW4nOgogICAgICAgIG9zLm1ha2VkaXJz
KGEub3V0LCBleGlzdF9vaz1UcnVlKQogICAgICAgIHRyYWluKGNmZywgYS5zZWVkLCBhLm91dCkK
ICAgIGVsc2U6CiAgICAgICAgd2l0aCBvcGVuKGEuaW5wdXRzKSBhcyBmOgogICAgICAgICAgICBp
ZHMgPSBqc29uLmxvYWQoZilbJ2lkcyddCiAgICAgICAgZGV2aWNlID0gJ2N1ZGEnIGlmIHRvcmNo
LmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAnY3B1JwogICAgICAgIG1vZGVsID0gbG9hZF9tb2Rl
bChjZmcsIGFkYXB0ZXI9YS5hZGFwdGVyKQogICAgICAgIGxwID0gcHJlZGljdChtb2RlbCwgaWRz
LCBjZmdbJ3BhZF9pZCddLCBjZmdbJ2V2YWxfYmF0Y2gnXSwgZGV2aWNlKQogICAgICAgIG5wLnNh
dmV6X2NvbXByZXNzZWQoYS5vdXQsIGxvZ3A9bHAuYXN0eXBlKG5wLmZsb2F0MzIpKQogICAgICAg
IGxvZyhmJ3ByZWRpY3RlZCB7bGVuKGlkcyl9IGlucHV0cyAtPiB7YS5vdXR9JykKCgppZiBfX25h
bWVfXyA9PSAnX19tYWluX18nOgogICAgbWFpbigpCg==
'''),
}
for name, (digest, blob) in CODE_FILES.items():
    data = base64.b64decode(blob)
    assert hashlib.sha256(data).hexdigest() == digest, f'{name} is corrupted'
    with open(os.path.join(CODE_DIR, name), 'wb') as f:
        f.write(data)
print('Code files written and verified:', ', '.join(CODE_FILES))
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import l2_posthoc
importlib.reload(l2_posthoc)
import l2_expert
importlib.reload(l2_expert)
import l2_llm as LL
importlib.reload(LL)
S = LL.stage_setup(CONFIG, CODE_DIR)

## 2. Downloads (checkpoints, MELD features, the 7B model about 15 GB), data checks and inputs (train and dev labels only)

In [ ]:
LL.stage_prepare(S)

## 3. Released ECERC checkpoints on dev

In [ ]:
LL.stage_ecerc_dev(S)

## 4. Fine-tune the LLM expert (2 seeds in parallel; the longest step)
Progress lines appear at each dev evaluation (every third of the epoch). Training is capped at 7 hours per seed.

In [ ]:
LL.stage_train_llm(S)

## 5. Dev-only fusion weight, cross-fitted forecast and gate

In [ ]:
LL.stage_dev_analysis(S)

## 6. Freeze

In [ ]:
LL.stage_freeze(S)

## 7. Test evaluation (runs once; skipped if the gate failed)

In [ ]:
LL.stage_test(S)

## 8. Results
Prints the summary and writes `ecerc_l2_llm_results.zip`. On Kaggle it's under **Output** (`ecerc_l2/ecerc_l2_llm_results.zip`).

In [ ]:
zpath = LL.stage_package(S)
try:
    from google.colab import files
    files.download(zpath)
except ImportError:
    print('Download this file from the Output panel ->', zpath)